# Comparing pretrained models for bean-leaf classification

We compare EfficientNet-B0, ResNet-18 and MobileNetV3-Large with frozen features and with the last feature block fine-tuned. All training uses the original training split; validation chooses the winner by accuracy. The test set is evaluated only after selection.

The shared implementation lives in `beans_transfer.py`. Run this notebook inside the repository with NumPy, Pillow, scikit-learn, PyTorch, torchvision, pandas and Jupyter installed. Pretrained weights are downloaded on first use if not already cached. Training is CPU-only and may take substantial time.

## 1. Configure the experiment

Set `RUN_TRAINING = True` to run all six experiments. The output directory must **not already exist**; use a new name for each run. To inspect an existing completed run instead, keep it `False` and set `run_directory` to that run's directory. The script's defaults use seed 42, batch size 16, up to 20 epochs, and patience 5.

In [1]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

relative_source = Path('session3_coding_with_a_copilot/HandsOn/src')
source_dir = next((p / relative_source for p in (Path.cwd(), *Path.cwd().parents)
                   if (p / relative_source / 'beans_transfer.py').is_file()), None)
if source_dir is None:
    raise FileNotFoundError('Start Jupyter within the repository so the source directory can be found.')
run_directory = source_dir / 'beans_results_run1'
RUN_TRAINING = False

In [2]:
if RUN_TRAINING:
    subprocess.run([sys.executable, str(source_dir / 'beans_transfer.py'), 'train',
                    '--output', str(run_directory)], check=True)
else:
    print(f'Training disabled. Reading an existing run from {run_directory}.')

Training disabled. Reading an existing run from /mnt/c/Users/edgar/Source/AI-Workshop-UNTRM/session3_coding_with_a_copilot/HandsOn/src/beans_results_run1.


## 2. Compare on validation

The six results are selected by validation accuracy, then macro F1, then frozen preference and a fixed model order. The majority baseline always predicts the class most common in training. Each checkpoint is selected by validation accuracy, with loss as the epoch-level tie-breaker. The reported elapsed time is CPU training wall time, not an inference benchmark.

In [3]:
validation_path = run_directory / 'validation_results.json'
if not validation_path.is_file():
    raise FileNotFoundError(f'No completed validation results at {validation_path}; run training first.')
with validation_path.open(encoding='utf-8') as file:
    validation = json.load(file)
rows = []
for result in validation['experiments']:
    metrics = result['validation']
    rows.append({'model': result['model'], 'variant': result['variant'],
                 'accuracy': metrics['accuracy'], 'macro_f1': metrics['macro_f1'],
                 **{f'{name}_recall': values['recall'] for name, values in metrics['per_class'].items()},
                 'training_seconds': result['training_seconds'],
                 'trainable_parameters': result['trainable_parameters'],
                 'checkpoint_bytes': result['checkpoint_bytes'], 'best_epoch': result['best_epoch']})
display(pd.DataFrame(rows).sort_values(['accuracy', 'macro_f1'], ascending=False))
print('Majority baseline:', validation['majority_class'], validation['majority_baseline']['accuracy'])
print('Selected using validation:', validation['winner'])

,model,variant,accuracy,macro_f1,angular_leaf_spot_recall,bean_rust_recall,healthy_recall,training_seconds,trainable_parameters,checkpoint_bytes,best_epoch
3,resnet18,fine_tuned,0.984962,0.985016,0.977273,0.977778,1.000000,369.714225,8390147,44792587,4
1,efficientnet_b0,fine_tuned,0.969925,0.970003,0.931818,0.977778,1.000000,681.207789,413443,16347481,13
0,efficientnet_b0,frozen,0.954887,0.955468,0.931818,0.977778,0.954545,887.137443,3843,16346017,20
5,mobilenet_v3_large,fine_tuned,0.947368,0.946827,0.886364,0.955556,1.000000,148.654469,157443,17033059,3
4,mobilenet_v3_large,frozen,0.932331,0.931682,0.840909,0.977778,0.977273,189.675482,3843,17031723,5
2,resnet18,frozen,0.924812,0.924526,0.886364,0.955556,0.931818,591.250712,1539,44792075,17


Majority baseline: bean_rust 0.3383458646616541
Selected using validation: {'model': 'resnet18', 'variant': 'fine_tuned', 'checkpoint': 'resnet18_fine_tuned.pt'}


## 3. Report the selected model on the test set

The script writes the validation selection before decoding or evaluating test images. The confusion matrix has true classes in rows and predictions in columns, in the class order shown below. The Wilson interval describes uncertainty in accuracy for this finite test sample; it does not address distribution shift or dependence between images.

In [4]:
with (run_directory / 'test_results.json').open(encoding='utf-8') as file:
    test_report = json.load(file)
if test_report['winner'] != validation['winner']:
    raise ValueError('Test results do not match the recorded validation winner.')
test = test_report['test']
print('Test accuracy:', test['accuracy'])
print('95% Wilson interval:', test['accuracy_95_percent_wilson_interval'])
print('Test macro F1:', test['macro_f1'])
display(pd.DataFrame(test['per_class']).T)
classes = list(test['per_class'])
display(pd.DataFrame(test['confusion_matrix'], index=classes, columns=classes))

Test accuracy: 0.9453125
95% Wilson interval: [0.8914150997002529, 0.9732597986186536]
Test macro F1: 0.9454382209460493


,precision,recall,f1,support
angular_leaf_spot,0.877551,1.000000,0.934783,43.0
bean_rust,0.973684,0.860465,0.913580,43.0
healthy,1.000000,0.976190,0.987952,42.0


,angular_leaf_spot,bean_rust,healthy
angular_leaf_spot,43,0,0
bean_rust,6,37,0
healthy,0,1,41


## 4. Interpret the results

Compare frozen and fine-tuned results for each backbone. Validation and test are small, so close differences may not generalize; repeated tuning on validation can also overfit it. Background, camera, lighting and near-duplicate images may inflate or reduce performance on future photographs. These three classes cannot represent every possible disease, and predictions are not a substitute for expert assessment. See the HandsOn README for checkpoint reuse and prediction.